# S3 Checkpoint Storage with TransformersTrainer

This notebook demonstrates how to configure **S3 storage for checkpoints** in distributed training using `TransformersTrainer` on Red Hat OpenShift AI.

## Overview

In this example, we fine-tune the **Qwen 2.5 1.5B Instruct** model on the **Stanford Alpaca** instruction-following dataset. The training runs on 2 GPU nodes with **checkpoints uploaded to S3-compatible object storage**, enabling automatic checkpoint saves during training and JIT (Just-In-Time) checkpointing on SIGTERM signals (e.g., during preemption, scaling, or graceful shutdown).

**Model:** Qwen 2.5 1.5B Instruct - A 1.5B parameter instruction-tuned model, small enough for quick demonstration while being powerful enough for real-world tasks.

**Dataset:** We use 500 samples from the Stanford Alpaca instruction-following dataset.

### What You'll Learn

| Feature | Description |
|---------|-------------|
| **S3 Checkpoint Storage** | Upload/download checkpoints to S3-compatible object storage (AWS S3, MinIO, etc.) |
| **JIT Checkpointing** | Automatically save training state on SIGTERM signal (preemption-safe) |
| **Periodic Checkpointing** | Configure regular checkpoint saves using `PeriodicCheckpointConfig` |
| **Auto-Resume** | Automatically resume training from the latest checkpoint in S3 on restart |
| **Data Connections** | Use OpenShift AI Data Connections to securely manage S3 credentials |

### Why S3 for Checkpoint Storage?

S3-compatible object storage offers several advantages for distributed training:

**Benefits:**
- **Scalability** - No storage size limits, pay only for what you use
- **Durability** - Built-in replication and high availability (99.999999999% durability for AWS S3)
- **Accessibility** - Access checkpoints from anywhere, share across clusters and regions
- **Cost-Effective** - Object storage is typically cheaper than block storage for large checkpoints
- **Preemption Safety** - JIT checkpointing ensures training state is saved when pods receive SIGTERM

**Common scenarios where S3 checkpoint storage is valuable:**
- **Spot/Preemptible instances** - Training on cost-effective but reclaimable instances
- **Kueue preemption** - Higher-priority workloads may preempt lower-priority jobs
- **Node maintenance** - Cluster upgrades or node drains
- **Resource pressure** - Pods evicted due to memory or resource limits
- **Long-running training** - Unlimited storage without size constraints
- **Cross-cluster training** - Share checkpoints across different clusters or regions

**When to use S3 vs PVC for checkpoints:**

| Scenario | Recommendation |
|----------|----------------|
| Large model checkpoints (>10GB) | **S3** - More cost-effective than PVC |
| Long-running training | **S3** - Unlimited storage, better durability |
| Cross-cluster training | **S3** - Share checkpoints across clusters |
| Spot/Preemptible instances | **S3 + JIT** - Preemption-safe with high durability |
| Short training runs (<1 hour) | **PVC** - Simpler setup, no S3 credentials needed |

S3 checkpoint storage ensures that:
1. Training pauses safely after the current optimizer step
2. Model state, optimizer state, and training progress are uploaded to S3
3. When the job restarts, training automatically resumes from the latest S3 checkpoint

### Prerequisites

Before running this notebook, ensure you have:

1. **OpenShift AI Cluster** with Kubeflow Trainer v2 enabled
2. **Workbench** running Python 3.12+ with GPU access
3. **S3-Compatible Storage** with a bucket created:
   - AWS S3, MinIO, or any S3-compatible object storage
   - S3 bucket created and accessible
   - S3 credentials (Access Key ID and Secret Access Key)
4. **Data Connection** in OpenShift AI:
   - Create via: OpenShift AI → Data Science Project → Data Connections → Add data connection
   - Required fields: AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, AWS_S3_ENDPOINT, AWS_S3_BUCKET, AWS_DEFAULT_REGION
5. **Shared PVC** (optional, for caching model/dataset downloads):
   - Name: `shared` with ReadWriteMany (RWX) access mode
   - Recommended Size: 20Gi
   - Mount Path: `/opt/app-root/src/shared` in the workbench

## Setup and Imports

Install the Kubeflow SDK and required packages.

In [ ]:
!python3 -m pip install --force-reinstall --no-cache-dir -U "kubeflow @ git+https://github.com/opendatahub-io/kubeflow-sdk.git"
!python3 -m pip install datasets transformers accelerate huggingface_hub

In [ ]:
import os

import kubeflow
import torch
from datasets import load_dataset
from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.trainer import TrainerClient
from kubeflow.trainer.rhai import TransformersTrainer
from kubeflow.trainer.rhai.transformers import PeriodicCheckpointConfig
from kubernetes import client as k8s
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"Kubeflow SDK version: {getattr(kubeflow, '__version__', 'unknown')}")
print("✅ All imports successful")

## Configuration

Configure authentication and paths.

### Environment Variables

The following environment variables are required for API authentication:
- `OPENSHIFT_API_URL` - Your OpenShift API server URL (e.g., `https://api.cluster.example.com:6443`)
- `NOTEBOOK_USER_TOKEN` - Authentication token for API access

**Note:** These are typically auto-set in OpenShift AI workbenches. If not set, uncomment and fill in the values in the cell below.

In [ ]:
# ============================================================================
# AUTHENTICATION CONFIGURATION
# ============================================================================
# These values are typically auto-set in OpenShift AI workbenches.
# If not set, uncomment and fill in your values below:

# api_server = "https://api.your-cluster.example.com:6443"
# token = "sha256~your-token-here"

# Try to get from environment variables first
api_server = os.getenv("OPENSHIFT_API_URL")
token = os.getenv("NOTEBOOK_USER_TOKEN")

if not api_server or not token:
    raise RuntimeError(
        "OPENSHIFT_API_URL and NOTEBOOK_USER_TOKEN environment variables are required.\n"
        "Either set them in your environment, or uncomment and fill in the values above."
    )

# Configure Kubernetes client
configuration = k8s.Configuration()
configuration.host = api_server
configuration.verify_ssl = False  # Set to True if using trusted certificates
configuration.api_key = {"authorization": f"Bearer {token}"}

# ============================================================================
# PVC CONFIGURATION FOR MODEL/DATA STORAGE
# ============================================================================
# Note: Checkpoints are saved to S3, not PVC. The PVC is only used for:
# - Storing the pre-downloaded model and dataset
# - Mounting to training pods so they can access model/data without internet
#
# NOTEBOOK_SHARED_PATH is where *your workbench* sees the PVC.
# This depends on what PVC you attached when you created the workbench.
# On OpenShift AI, the default mount convention is:
#   /opt/app-root/src/<pvc-name>
#
# If your PVC is not named "shared", set PVC_NAME accordingly.
PVC_NAME = "shared"
NOTEBOOK_SHARED_PATH = f"/opt/app-root/src/{PVC_NAME}"

# Quick sanity check to help users discover the right workbench mount
if not os.path.exists(NOTEBOOK_SHARED_PATH):
    print(
        "⚠️  Expected workbench PVC mount not found at: "
        f"{NOTEBOOK_SHARED_PATH}\n"
        "If your PVC has a different name or mount, update PVC_NAME/NOTEBOOK_SHARED_PATH.\n"
        "Tip: in a workbench, PVCs are typically under /opt/app-root/src/."
    )

# Model Configuration
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# Paths for notebook operations only (downloading model/data to PVC)
# Note: Training pods will mount this PVC to access model/data
# Checkpoints are saved to S3 (configured via output_dir="s3://..." parameter)
NOTEBOOK_MODEL_PATH = f"{NOTEBOOK_SHARED_PATH}/models/qwen2.5-1.5b-instruct"
NOTEBOOK_DATA_PATH = f"{NOTEBOOK_SHARED_PATH}/data/alpaca_processed"

print(f"API Server: {api_server}")
print(f"Model: {MODEL_NAME}")
print(f"PVC name: {PVC_NAME}")
print(f"Workbench PVC mount: {NOTEBOOK_SHARED_PATH}")
print(f"Notebook Model Path: {NOTEBOOK_MODEL_PATH}")
print(f"Notebook Data Path: {NOTEBOOK_DATA_PATH}")
print("\n💾 Note: Checkpoints will be saved to S3, not PVC")

## Download Model and Dataset to Shared PVC

Before submitting the training job, we pre-download the model and dataset to the shared PVC. This ensures:
- **Offline Training:** Training pods don't need internet access during training
- **Faster Startup:** No download delays when training pods start
- **Consistency:** All nodes use the same model weights and data

In [ ]:
from huggingface_hub import snapshot_download

# Download model to PVC
if os.path.exists(NOTEBOOK_MODEL_PATH) and os.listdir(NOTEBOOK_MODEL_PATH):
    print(f"✅ Model already exists at {NOTEBOOK_MODEL_PATH}")
    # Verify model files
    model_files = [
        f
        for f in os.listdir(NOTEBOOK_MODEL_PATH)
        if f.endswith((".safetensors", ".bin"))
    ]
    print(f"📁 Model weight files: {model_files}")
else:
    print(f"🔄 Downloading model {MODEL_NAME} to {NOTEBOOK_MODEL_PATH}...")
    print("   This uses direct file download (no loading into memory)")
    os.makedirs(NOTEBOOK_MODEL_PATH, exist_ok=True)

    try:
        # Direct download - MUCH faster than loading the model
        # Downloads files directly without loading 1.5B parameters into memory
        snapshot_download(
            repo_id=MODEL_NAME,
            local_dir=NOTEBOOK_MODEL_PATH,
            local_dir_use_symlinks=False,  # Copy files instead of symlinks
            resume_download=True,  # Resume if interrupted
            token=None,  # Add HF token if model requires auth
        )

        print(f"✅ Model downloaded to {NOTEBOOK_MODEL_PATH}")

        # Verify download was successful
        all_files = os.listdir(NOTEBOOK_MODEL_PATH)
        model_files = [f for f in all_files if f.endswith((".safetensors", ".bin"))]
        config_files = [f for f in all_files if "config.json" in f]
        tokenizer_files = [f for f in all_files if "tokenizer" in f]

        print("📁 Downloaded files:")
        print(f"   - Model weights: {model_files}")
        print(f"   - Config files: {config_files}")
        print(f"   - Tokenizer files: {len(tokenizer_files)} files")

        if not model_files:
            raise FileNotFoundError(
                "❌ No model weight files (.safetensors or .bin) found after download!"
            )
        if not config_files:
            raise FileNotFoundError("❌ No config.json found after download!")

        print("✅ All required files verified!")

    except Exception as e:
        print(f"❌ Error downloading model: {e}")
        print("   Cleaning up partial download...")
        import shutil

        if os.path.exists(NOTEBOOK_MODEL_PATH):
            shutil.rmtree(NOTEBOOK_MODEL_PATH)
        raise

In [ ]:
# Download and prepare dataset
if os.path.exists(NOTEBOOK_DATA_PATH) and os.listdir(NOTEBOOK_DATA_PATH):
    print(f"✅ Dataset already exists at {NOTEBOOK_DATA_PATH}")
else:
    print("🔄 Downloading and processing Alpaca dataset...")
    os.makedirs(NOTEBOOK_DATA_PATH, exist_ok=True)

    try:
        # Load subset of Alpaca dataset
        print("   Downloading Alpaca dataset from HuggingFace...")
        dataset = load_dataset("tatsu-lab/alpaca", split="train[:500]")
        print(f"   Loaded {len(dataset)} examples")

        # Load tokenizer for preprocessing
        print("   Loading tokenizer...")
        tokenizer = AutoTokenizer.from_pretrained(
            NOTEBOOK_MODEL_PATH, use_fast=True, trust_remote_code=True
        )
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        print(f"   Tokenizer loaded (vocab size: {len(tokenizer)})")

        def format_instruction(example):
            if example.get("input"):
                text = f"### Instruction:\n{example['instruction']}\n\n### Input:\n{example['input']}\n\n### Response:\n{example['output']}"
            else:
                text = f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"
            return {"text": text}

        print("   Formatting instructions...")
        dataset = dataset.map(format_instruction, remove_columns=dataset.column_names)

        def tokenize_function(examples):
            tokenized = tokenizer(
                examples["text"],
                padding="max_length",
                truncation=True,
                max_length=512,
            )
            tokenized["labels"] = tokenized["input_ids"].copy()
            return tokenized

        print("   Tokenizing dataset (this may take a minute)...")
        tokenized_dataset = dataset.map(
            tokenize_function, batched=True, remove_columns=["text"]
        )

        print(f"   Saving tokenized dataset to {NOTEBOOK_DATA_PATH}...")
        tokenized_dataset.save_to_disk(NOTEBOOK_DATA_PATH)
        print(f"✅ Dataset saved to {NOTEBOOK_DATA_PATH}")

        # Verify
        saved_files = os.listdir(NOTEBOOK_DATA_PATH)
        print(f"📁 Dataset files: {saved_files}")

    except Exception as e:
        print(f"❌ Error preparing dataset: {e}")
        import shutil

        if os.path.exists(NOTEBOOK_DATA_PATH):
            shutil.rmtree(NOTEBOOK_DATA_PATH)
        raise

print("\n✅ Model and dataset ready on PVC!")

## Define the Training Function

The training function runs inside each training pod as a distributed PyTorch process. TransformersTrainer serializes this function and executes it via `torchrun` on each node.

### Training Configuration

| Parameter | Value | Description |
|-----------|-------|-------------|
| `num_train_epochs` | 5 | Multiple epochs to allow time for testing pause/resume |
| `per_device_train_batch_size` | 2 | Samples per GPU per step |
| `gradient_accumulation_steps` | 4 | Effective batch size = 2 × 4 × 2 nodes = 16 |
| `learning_rate` | 2e-5 | Standard fine-tuning rate |
| `save_steps` | 20 | Checkpoint every 20 steps |
| `bf16` | True | Use bfloat16 mixed precision |

### Key Points

- **Supported Trainers:** Use `transformers.Trainer` or `trl.SFTTrainer` - both are auto-instrumented
- **No Manual Setup:** JIT checkpointing and progress tracking callbacks are injected automatically
- **Local Files Only:** Model and data are loaded from the mounted PVC (no network access needed)

In [ ]:
def train_func():
    """SFT training function using HuggingFace Trainer.

    TransformersTrainer automatically:
    - Injects JIT checkpoint handler for SIGTERM (preemption-safe)
    - Injects KubeflowProgressCallback for real-time metrics
    - Auto-resumes from the latest S3 checkpoint when available
    - Configures S3 upload/download for checkpoints
    """
    import json
    import os

    import torch
    from datasets import load_from_disk
    from transformers import (
        AutoModelForCausalLM,
        DataCollatorForLanguageModeling,
        PreTrainedTokenizerFast,
        Trainer,
        TrainingArguments,
    )

    rank = int(os.environ.get("RANK", 0))
    local_rank = int(os.environ.get("LOCAL_RANK", 0))

    # Model/data are on the shared PVC mounted at /mnt/shared (via pod_template_overrides)
    model_path = "/mnt/shared/models/qwen2.5-1.5b-instruct"
    data_path = "/mnt/shared/data/alpaca_processed"

    print(f"🚀 Starting training on rank {rank}")

    if torch.cuda.is_available():
        torch.cuda.set_device(local_rank)
        print(f"🔧 GPU: {torch.cuda.get_device_name(local_rank)}")

    # Load tokenizer directly from tokenizer.json file
    # This bypasses AutoTokenizer's hub validation that fails with local paths
    print(f"📥 Loading tokenizer from: {model_path}")
    tokenizer_file = os.path.join(model_path, "tokenizer.json")
    tokenizer_config_file = os.path.join(model_path, "tokenizer_config.json")

    # Load tokenizer config to get special tokens
    with open(tokenizer_config_file) as f:
        tokenizer_config = json.load(f)

    tokenizer = PreTrainedTokenizerFast(
        tokenizer_file=tokenizer_file,
        eos_token=tokenizer_config.get("eos_token", "<|endoftext|>"),
        pad_token=tokenizer_config.get("pad_token"),
        bos_token=tokenizer_config.get("bos_token"),
        unk_token=tokenizer_config.get("unk_token"),
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Load model directly from local path
    print(f"📥 Loading model from: {model_path}")
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.bfloat16,
        device_map={"": local_rank},
        local_files_only=True,
        trust_remote_code=True,
    )

    # Load dataset
    print(f"📥 Loading dataset from: {data_path}")
    tokenized_dataset = load_from_disk(data_path)

    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
    )

    # TransformersTrainer automatically configures:
    # - output_dir: Set from the s3:// URI (SDK handles S3 upload/download)
    # - save_strategy, save_steps, save_total_limit: Set from PeriodicCheckpointConfig
    # - S3 credentials: Injected from data_connection_name secret
    training_args = TrainingArguments(
        output_dir="/tmp/output",  # Placeholder - SDK overrides this with S3-backed path
        num_train_epochs=5,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-5,
        logging_steps=5,
        report_to="none",
        bf16=True,
        ddp_find_unused_parameters=False,
    )

    # Trainer - TransformersTrainer injects:
    # - JIT checkpoint handler for SIGTERM (saves to S3)
    # - KubeflowProgressCallback for real-time metrics
    # - S3 upload callback (uploads checkpoints asynchronously)
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset,
        data_collator=data_collator,
    )

    print(f"💾 Trainer output_dir: {trainer.args.output_dir}")

    # Train - auto-resumes from latest S3 checkpoint if available
    trainer.train()

    # Save final model (only on rank 0) - automatically uploaded to S3
    if rank == 0:
        final_path = os.path.join(trainer.args.output_dir, "final")
        os.makedirs(final_path, exist_ok=True)
        trainer.save_model(final_path)
        tokenizer.save_pretrained(final_path)
        print(f"✅ Final model saved to {final_path}")
        print("📤 Final model will be uploaded to S3")

    print(f"✅ Training complete on rank {rank}")


print("✅ Training function defined")

## Create the Trainer Client

Initialize the TrainerClient with authentication configuration.

In [ ]:
# Create client with authentication
api_client = k8s.ApiClient(configuration)

backend_config = KubernetesBackendConfig(
    client_configuration=api_client.configuration,
)

client = TrainerClient(backend_config)
print("✅ TrainerClient created")

# Get the torch-distributed runtime
runtime = client.backend.get_runtime("torch-distributed")
print(f"✅ Using runtime: {runtime.name}")

## Configure PVC Mount for Model and Data Access

### Why Do We Need to Mount the PVC?

When using S3 for checkpoint storage (`output_dir="s3://..."`), the SDK **does not automatically mount any PVC** to the training pods. This is different from PVC-based checkpointing (`output_dir="pvc://..."`), where the SDK automatically mounts the PVC.

**Key Difference:**

| Checkpoint Storage | PVC Mounting Behavior |
|-------------------|----------------------|
| `output_dir="pvc://shared/..."` | ✅ SDK **automatically** mounts PVC `shared` to training pods |
| `output_dir="s3://bucket/..."` | ❌ SDK does **NOT** automatically mount any PVC |

### Our Setup

In this example:
- ✅ **Checkpoints** → Saved to **S3** (via `output_dir="s3://..."`)
- ✅ **Model and Data** → Pre-downloaded to **PVC** (in your workbench)

**The Problem:**
- Your **workbench** has the PVC mounted at `/opt/app-root/src/shared` ✅
- But **training pods** are separate Kubernetes pods - they don't inherit your workbench's PVC mounts ❌
- Training function tries to load model from `/mnt/shared/models/...` → **will fail without PVC mount**

**The Solution:**
We need to **explicitly configure** the PVC mount for training pods using `PodTemplateOverrides`. This tells the SDK:
- Mount PVC named `shared` to all training pods
- Make it available at `/mnt/shared` 
- Training pods can now access pre-downloaded model and data

### Alternative Approach (No PVC)

If you don't want to use PVC mounting, you could:
- Remove the `pod_template_overrides` configuration
- Modify training function to download model directly from HuggingFace

In [ ]:
from kubeflow.trainer.options.kubernetes import (
    ContainerOverride,
    PodSpecOverride,
    PodTemplateOverride,
    PodTemplateOverrides,
)

# Configure pod template to mount the shared PVC for model/data access
# Note: This PVC is only for accessing pre-downloaded model and dataset
# Checkpoints are saved to S3, not to this PVC
pod_template_overrides = PodTemplateOverrides(
    PodTemplateOverride(
        target_jobs=["node"],  # Apply to all worker nodes
        spec=PodSpecOverride(
            volumes=[
                {"name": "shared", "persistentVolumeClaim": {"claimName": "shared"}}
            ],
            containers=[
                ContainerOverride(
                    name="node",
                    volume_mounts=[{"name": "shared", "mountPath": "/mnt/shared"}],
                )
            ],
        ),
    )
)

## Submit the Training Job with S3 Checkpoint Storage

Now we create and submit the distributed training job with S3 checkpoint storage enabled.

### Prerequisites: Create S3 Data Connection

Before submitting the job, you must create an S3 Data Connection in OpenShift AI:

1. **Navigate to Connections:**
   - Go: OpenShift AI Dashboard → Your Project → **Connections** tab
   
2. **Create Connection:**
   - Click "**Create connection**" button (top right)
   - In the dialog, select connection type: **S3 compatible object storage - v1**
   
3. **Fill in Connection Details:**
   
   **Required Fields:**
   - **Connection name**: A unique name for this connection (e.g., `s3-storage-connection`)
   - **Access key**: Your S3 access key ID
   - **Secret key**: Your S3 secret access key (will be hidden)
   - **Endpoint**: S3 endpoint URL
     - AWS S3: `https://s3.amazonaws.com` or `https://s3.<region>.amazonaws.com`
     - MinIO: `http://minio.example.com:9000`
   
   **Optional Fields:**
   - **Connection description**: Optional description for this connection
   - **Region**: S3 region (e.g., `us-east-1`)
   - **Bucket**: Default S3 bucket name (can be overridden in code)

4. **Save the Connection:**
   - Click "**Create**" button
   - This creates a Kubernetes secret with your S3 credentials

### Job Configuration

| Parameter | Value | Description |
|-----------|-------|-------------|
| `num_nodes` | 2 | Number of GPU nodes for distributed training |
| `nvidia.com/gpu` | 1 | GPUs per node |
| `cpu` | 4 | CPU cores per node |
| `memory` | 16Gi | Memory per node |

### S3 Checkpoint Storage Configuration

| Parameter | Value | Description |
|-----------|-------|-------------|
| `enable_jit_checkpoint` | `True` | Save checkpoint on SIGTERM (preemption) |
| `periodic_checkpoint_config` | See below | Configure periodic checkpoint saves |
| `output_dir` | `s3://<bucket>/<path>` | S3 URI for checkpoint storage |
| `data_connection_name` | `s3-storage-connection` | Name of S3 Data Connection (Kubernetes secret) |
| `verify_cloud_storage_ssl` | `True`/`False` | Verify SSL certificates for S3 connection |
| `verify_cloud_storage_access` | `True`/`False` | Pre-validate S3 access before job starts |

### How S3 Checkpoint Storage Works

When `output_dir="s3://..."` is configured:

1. **S3 Credentials Injection:** SDK loads credentials from the Data Connection secret
2. **Checkpoint Upload:** Checkpoints are saved locally first, then uploaded to S3 asynchronously
3. **JIT Checkpointing (if enabled):** When SIGTERM is received, training pauses after the current optimizer step and saves checkpoint to S3
4. **Auto-Resume:** On restart, the latest checkpoint is downloaded from S3 and training resumes

### S3 URI Format

The `output_dir` parameter uses S3 URI format to specify where checkpoints should be stored:

```
s3://<bucket-name>/<prefix-path>
```

**Breaking down the S3 URI:**

| Component | Description | Example |
|-----------|-------------|---------|
| `s3://` | Protocol prefix (required) | `s3://` |
| `<bucket-name>` | S3 bucket name (must exist) | `kubeflow-checkpoints` |
| `<prefix-path>` | Path within bucket (created automatically) | `experiments/qwen-alpaca` |

**Complete Example:**
```python
output_dir="s3://kubeflow-checkpoints/experiments/qwen-alpaca"
```

This configuration means:
- **Bucket:** `kubeflow-checkpoints` (must already exist in your S3 storage)
- **Prefix:** `experiments/qwen-alpaca/` (subdirectory created automatically if it doesn't exist)
- **Checkpoint locations:**
  - `s3://kubeflow-checkpoints/experiments/qwen-alpaca/checkpoint-20/`
  - `s3://kubeflow-checkpoints/experiments/qwen-alpaca/checkpoint-40/`
  - `s3://kubeflow-checkpoints/experiments/qwen-alpaca/final/`

**What This Example Uses:**

In this notebook, we use:
```python
output_dir="s3://kubeflow-checkpoints/checkpoints"
```

- Bucket: `kubeflow-checkpoints`
- Prefix: `checkpoints/`
- All checkpoints are stored under `s3://kubeflow-checkpoints/checkpoints/`

### SSL Verification

**`verify_cloud_storage_ssl` parameter:**

| Value | Use Case |
|-------|----------|
| `True` | Production AWS S3 with valid certificates |
| `False` | Self-signed certs (MinIO) or internal S3 without valid certs |

**Security Note:** Always use `True` for production AWS S3. Only use `False` for development/testing with self-signed certificates.

### S3 Access Validation

**`verify_cloud_storage_access` parameter:**

| Value | Behavior |
|-------|----------|
| `True` | SDK validates S3 connectivity **before** creating training pods (slower, safer) |
| `False` | Job submits immediately without testing S3 (faster, fails during training if S3 inaccessible) |

**Recommendation:** Use `True` for first-time setup to catch S3 configuration errors early.

### Periodic Checkpointing

In addition to JIT checkpointing, you can configure periodic saves:

```python
PeriodicCheckpointConfig(
    save_strategy="steps",  # or "epoch"
    save_steps=20,           # Save every 50 steps
    save_total_limit=2,      # Keep only 2 most recent checkpoints
)
```

> **Note:** Checkpoint saves (periodic or JIT) involve uploading to S3, which can take time depending on checkpoint size and network speed. Avoid checkpointing too frequently (e.g., every step) as this can significantly increase total training time.

In [ ]:
# Configure periodic checkpointing - SDK injects these into TrainingArguments
checkpoint_config = PeriodicCheckpointConfig(
    save_strategy="steps",
    save_steps=20,
    save_total_limit=2,
)

# Create TransformersTrainer with S3 checkpointing enabled
# The output_dir="s3://kubeflow-checkpoints/checkpoints" tells the SDK to:
# - Inject S3 credentials from the data_connection_name secret
# - Configure S3 filesystem for checkpoint upload/download
# - Set TrainingArguments.output_dir to a local path that syncs with S3
# - Enable JIT checkpointing (saves state to S3 on SIGTERM)
# - Auto-resume from latest S3 checkpoint on restart
trainer = TransformersTrainer(
    func=train_func,
    num_nodes=2,
    resources_per_node={
        "nvidia.com/gpu": 1,
        "cpu": "4",
        "memory": "16Gi",
    },
    # Make the training function cleaner: set offline mode at the pod level
    env={
        "HF_HUB_OFFLINE": "1",
        "TRANSFORMERS_OFFLINE": "1",
    },
    packages_to_install=["fsspec", "s3fs"],
    # JIT Checkpointing: Save checkpoint on SIGTERM (preemption)
    enable_jit_checkpoint=True,
    # Periodic Checkpointing: Save checkpoint every save_steps
    periodic_checkpoint_config=checkpoint_config,
    # S3 URI format: s3://<bucket-name>/<path>
    # - Bucket: "kubeflow-checkpoints" (must exist and be accessible)
    # - Path: "checkpoints" (created automatically)
    # All checkpoints (periodic + JIT) are saved to S3 at this location
    output_dir="s3://kubeflow-checkpoints/checkpoints",
    # Name of the Kubernetes secret containing S3 credentials (created via OpenShift AI → Project → Data Connections)
    # Must contain: AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, AWS_S3_ENDPOINT, AWS_S3_BUCKET, AWS_DEFAULT_REGION
    data_connection_name="s3-storage-connection",
    # Disable SSL verification for S3 (use False for self-signed certs like MinIO,
    # True for production AWS S3)
    verify_cloud_storage_ssl=False,
    # Skip pre-flight S3 validation (False = faster submission but fails during training
    # if S3 inaccessible, True = validates before job starts)
    verify_cloud_storage_access=True,
)

print("✅ TransformersTrainer configured with:")
print("   - JIT Checkpointing: ENABLED (saves to S3 on SIGTERM)")
print("   - Periodic Checkpointing: Every 20 steps to S3, keep 2 most recent")
print("   - Auto-Resume: ENABLED (resumes from latest S3 checkpoint)")
print("   - S3 URI: s3://kubeflow-checkpoints/checkpoints")

# Submit the training job
job_name = client.train(
    trainer=trainer, runtime=runtime, options=[pod_template_overrides]
)
print(f"\n✅ Training job submitted: {job_name}")
print("💾 Checkpoints will be saved to S3: s3://kubeflow-checkpoints/checkpoints")

## Get Job Status

Check the final status of the training job after completion.

In [ ]:
# Check job status
job = client.get_job(job_name)
print("Final TrainJob Status:")
print(f"   Name: {job.name}")
print(f"   Status: {job.status}")
print(f"   Created: {job.creation_timestamp}")
print(f"   Nodes: {job.num_nodes}")
print(f"   Runtime: {job.runtime.name}")

if job.steps:
    print("   Steps:")
    for step in job.steps:
        print(f"     - {step.name}: {step.status}")
    print()

In [ ]:
# Configure S3 credentials for downloading checkpoints
# These should match the credentials you used in your Data Connection
# Update these values with your actual S3 credentials

# S3 Endpoint and Region
S3_ENDPOINT_URL = "your-endpoint-url"  # Replace with your S3 endpoint (e.g., MinIO: http://minio.example.com:9000)
S3_REGION = "your-region"  # Replace with your S3 region

# S3 Credentials
AWS_ACCESS_KEY_ID = "your-access-key-id"  # Replace with your S3 access key
AWS_SECRET_ACCESS_KEY = "your-secret-access-key"  # Replace with your S3 secret key

# Checkpoint location - should match what you used in TransformersTrainer
S3_OUTPUT_DIR = (
    "s3://kubeflow-checkpoints/checkpoints"  # Match your output_dir parameter
)

# Parse bucket and prefix from the S3 URI
s3_parts = S3_OUTPUT_DIR.replace("s3://", "").split("/", 1)
S3_CHECKPOINT_BUCKET = s3_parts[0]
S3_CHECKPOINT_PREFIX = s3_parts[1] if len(s3_parts) > 1 else ""

print("✅ S3 credentials configured - ready to download checkpoints")

## Verify Checkpoints in S3

After training completes (or after a preemption/restart), you can verify the checkpoints saved in S3.

### Configure S3 Credentials

The cell below sets up S3 credentials to access your checkpoints. You need to update the following values:

**Required Configuration:**
- `S3_ENDPOINT_URL` - Your S3 endpoint URL
  - AWS S3: `https://s3.amazonaws.com` or `https://s3.<region>.amazonaws.com`
  - MinIO: `http://minio.example.com:9000`
- `S3_REGION` - Your S3 region (e.g., `us-east-1`)
- `AWS_ACCESS_KEY_ID` - Your S3 access key
- `AWS_SECRET_ACCESS_KEY` - Your S3 secret key
- `S3_OUTPUT_DIR` - Should match the `output_dir` you used in `TransformersTrainer`

**Important:** These credentials should match what you configured in your Data Connection for the training job.

### ⚠️ Security Best Practices

**IMPORTANT:** Follow these security practices when working with S3 credentials:

1. **Clear outputs before sharing** - Use "Cell → All Output → Clear" in Jupyter before sharing notebooks
2. **Don't hardcode credentials in production** - Use environment variables or secret management systems
3. **Use IAM roles when possible** - On AWS, use IAM roles for service accounts instead of access keys
4. **Rotate credentials regularly** - Change S3 access keys periodically
5. **Limit permissions** - Grant only necessary S3 permissions (read/write to specific bucket/prefix)

### Checkpoint Structure in S3

The training function saves checkpoints with this structure:
```
s3://kubeflow-checkpoints/checkpoints/
├── checkpoint-<step>/  # Intermediate checkpoints (saved every save_steps)
├── checkpoint-<N>/     # Checkpoint at final step (N = last step number)
└── final/              # Final merged model ready for inference
```

**Example:** With the configuration in this notebook (`output_dir="s3://kubeflow-checkpoints/checkpoints"`):
- Bucket: `kubeflow-checkpoints`
- Prefix: `checkpoints/`
- Intermediate checkpoints: `s3://kubeflow-checkpoints/checkpoints/checkpoint-20/`, `checkpoint-40/`, etc.
- Final model: `s3://kubeflow-checkpoints/checkpoints/final/`

## Test the Trained Model (Optional)

After training completes, you can download the fine-tuned model from S3 and test it locally.

**What this section does:**
1. Downloads the final model from S3 location: `s3://{S3_CHECKPOINT_BUCKET}/{S3_CHECKPOINT_PREFIX}/final/`
2. Loads the model into memory (requires GPU)
3. Tests the model with a sample prompt

**Requirements:**
- Training must be completed (final model saved to S3)
- GPU available in your workbench (for model inference)
- S3 credentials configured in the previous cell

In [ ]:
# Install boto3 if not already installed
!pip install boto3 -q

import os

import boto3
import urllib3

# Suppress SSL warnings when using self-signed certificates
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


def download_checkpoint_from_s3(
    s3_bucket,
    s3_prefix,
    local_path,
    s3_endpoint,
    s3_region,
    access_key,
    secret_key,
    verify_ssl=True,
):
    """Download checkpoint from S3 to local path.

    Args:
        s3_bucket: S3 bucket name
        s3_prefix: S3 prefix/path within bucket
        local_path: Local directory to download files to
        s3_endpoint: S3 endpoint URL
        s3_region: S3 region
        access_key: AWS access key ID
        secret_key: AWS secret access key
        verify_ssl: Whether to verify SSL certificates (set False for self-signed certs)
    """
    print(f"📥 Downloading checkpoint from s3://{s3_bucket}/{s3_prefix}")
    print(f"   to local path: {local_path}")
    print(
        f"   SSL verification: {'Enabled' if verify_ssl else 'Disabled (self-signed cert)'}"
    )

    # Create S3 client with credentials
    s3_client = boto3.client(
        "s3",
        endpoint_url=s3_endpoint,
        region_name=s3_region,
        aws_access_key_id=access_key,
        aws_secret_access_key=secret_key,
        verify=verify_ssl,  # Disable SSL verification for self-signed certs
    )

    # Create local directory
    os.makedirs(local_path, exist_ok=True)

    # List all files under the S3 prefix
    response = s3_client.list_objects_v2(Bucket=s3_bucket, Prefix=s3_prefix + "/")

    if "Contents" not in response:
        raise FileNotFoundError(f"No files found in s3://{s3_bucket}/{s3_prefix}")

    # Download each file
    file_count = 0
    for obj in response["Contents"]:
        s3_key = obj["Key"]
        # Skip if it's just a directory marker
        if s3_key.endswith("/"):
            continue

        # Get relative path (remove prefix)
        relative_path = s3_key[len(s3_prefix) + 1 :]
        if not relative_path:  # Skip the directory itself
            continue

        local_file = os.path.join(local_path, relative_path)

        # Create subdirectories if needed
        os.makedirs(os.path.dirname(local_file), exist_ok=True)

        # Download file
        print(f"   Downloading: {s3_key}")
        s3_client.download_file(s3_bucket, s3_key, local_file)
        file_count += 1

    print(f"✅ Downloaded {file_count} files to {local_path}")
    return local_path


print("✅ S3 download utility defined")

In [ ]:
# Download the final model from S3
S3_FINAL_MODEL_PREFIX = f"{S3_CHECKPOINT_PREFIX}/final"
LOCAL_DOWNLOAD_PATH = "/tmp/downloaded_model"

print("📍 Downloading from S3:")
print(f"   Bucket: {S3_CHECKPOINT_BUCKET}")
print(f"   Prefix: {S3_FINAL_MODEL_PREFIX}")
print(f"   Full S3 URI: s3://{S3_CHECKPOINT_BUCKET}/{S3_FINAL_MODEL_PREFIX}")

# Download checkpoint from S3 (using credentials from the Data Connection secret)
final_checkpoint = download_checkpoint_from_s3(
    s3_bucket=S3_CHECKPOINT_BUCKET,
    s3_prefix=S3_FINAL_MODEL_PREFIX,
    local_path=LOCAL_DOWNLOAD_PATH,
    s3_endpoint=S3_ENDPOINT_URL,
    s3_region=S3_REGION,
    access_key=AWS_ACCESS_KEY_ID,
    secret_key=AWS_SECRET_ACCESS_KEY,
    verify_ssl=False,
)

print(f"\n📂 Loading checkpoint from: {final_checkpoint}")

# Load the model and tokenizer
trained_tokenizer = AutoTokenizer.from_pretrained(
    final_checkpoint, trust_remote_code=True
)
trained_model = AutoModelForCausalLM.from_pretrained(
    final_checkpoint,
    torch_dtype=torch.bfloat16,
    device_map="cuda:0",
    trust_remote_code=True,
)

print("✅ Model loaded successfully")
print(f"📊 Model parameters: {trained_model.num_parameters():,}")

# Test the model
test_prompt = "### Instruction:\nExplain what machine learning is in one sentence.\n\n### Response:"

print("\n📝 Testing model with prompt:")
print(test_prompt)
print("\n🤖 Model response:")

inputs = trained_tokenizer(test_prompt, return_tensors="pt").to(trained_model.device)

# Remove token_type_ids if present (not used by some models like Qwen)
if "token_type_ids" in inputs:
    del inputs["token_type_ids"]

with torch.no_grad():
    outputs = trained_model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7,
        top_p=0.95,
    )

response = trained_tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response.replace(test_prompt, "").strip())

print("\n✅ Model test completed!")

## Cleanup

Delete the training job and free resources.

**Note:** S3 checkpoints will remain in your S3 bucket. To delete them:

```python
# Optional: Delete checkpoints from S3
import boto3
s3_client = boto3.client('s3', endpoint_url=S3_ENDPOINT_URL, region_name=S3_REGION)
response = s3_client.list_objects_v2(Bucket=S3_CHECKPOINT_BUCKET, Prefix=S3_CHECKPOINT_PREFIX)
if 'Contents' in response:
    for obj in response['Contents']:
        s3_client.delete_object(Bucket=S3_CHECKPOINT_BUCKET, Key=obj['Key'])
    print(f"✅ Deleted checkpoints from s3://{S3_CHECKPOINT_BUCKET}/{S3_CHECKPOINT_PREFIX}")
```

In [ ]:
# Delete the training job
client.delete_job(name=job_name)
print(f"✅ Job {job_name} deleted")

In [ ]:
import gc

# Clear CUDA cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

gc.collect()
print("✅ Resources freed, CUDA cache cleared")

## Summary

Congratulations! You've successfully completed a distributed fine-tuning job with S3 checkpoint storage on OpenShift AI.

### What You Accomplished

| Step | Description |
|------|-------------|
| ✅ S3 Data Connection | Created S3 Data Connection in OpenShift AI for secure credential management |
| ✅ Model Download | Downloaded Qwen 2.5 1.5B Instruct to shared PVC (for training pods) |
| ✅ Dataset Preparation | Processed Stanford Alpaca dataset for instruction-tuning |
| ✅ Distributed Training | Ran 2-node distributed training with PyTorch DDP |
| ✅ S3 Checkpoint Storage | Saved all checkpoints to S3-compatible object storage |
| ✅ JIT Checkpointing | Enabled automatic checkpoint saves on SIGTERM |
| ✅ Periodic Checkpointing | Configured regular checkpoint saves every 20 steps |
| ✅ Auto-Resume | Training can resume from latest S3 checkpoint on restart |
| ✅ Model Testing | Downloaded final model from S3 and tested locally |

### Key Takeaways

1. **S3 Checkpoint Storage** offers scalability and durability:
   - Use `output_dir="s3://<bucket>/<prefix>"` for S3 storage
   - Unlimited storage capacity (pay-per-use)
   - Built-in replication and high availability
   - Access checkpoints from anywhere

2. **Data Connections** provide secure credential management:
   - Create in OpenShift AI → Data Science Project → Connections
   - Credentials stored as Kubernetes secrets
   - No hardcoded credentials in training job code

3. **JIT Checkpointing** makes training preemption-safe:
   - Enable with `enable_jit_checkpoint=True`
   - Automatically saves state when pod receives SIGTERM
   - Checkpoints uploaded to S3 asynchronously

4. **Periodic Checkpointing** provides regular saves:
   - Configure with `PeriodicCheckpointConfig`
   - Control save frequency (`save_strategy`, `save_steps`)
   - Limit storage usage (`save_total_limit`)

5. **Auto-Resume** minimizes training loss:
   - Training automatically resumes from latest S3 checkpoint
   - Incomplete checkpoints are detected and cleaned up
   - No manual intervention required

### TransformersTrainer S3 Checkpoint Storage Reference

| Parameter | Description | Example |
|-----------|-------------|---------|
| `output_dir` | S3 URI for checkpoints | `"s3://bucket/prefix"` |
| `data_connection_name` | S3 Data Connection name | `"s3-storage-connection"` |
| `enable_jit_checkpoint` | Save on SIGTERM | `True` |
| `periodic_checkpoint_config` | Configure periodic saves | `PeriodicCheckpointConfig(...)` |
| `verify_cloud_storage_ssl` | Verify SSL certificates | `True` for AWS, `False` for MinIO |
| `verify_cloud_storage_access` | Pre-validate S3 access | `True` for first-time setup |

### Resources

- [Kubeflow Trainer Documentation](https://www.kubeflow.org/docs/components/trainer/)
- [HuggingFace Transformers](https://huggingface.co/docs/transformers/)
- [Stanford Alpaca Dataset](https://huggingface.co/datasets/tatsu-lab/alpaca)
- [Boto3 S3 Documentation](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/s3.html)